# YOLOV11 — Benchmark on Indoor Dataset (7 classes)


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

IS_KAGGLE = os.path.exists("/kaggle/input")

def _safe_cuda_empty_cache():
    if not torch.cuda.is_available(): return
    try: torch.cuda.synchronize()
    except: pass
    try: torch.cuda.empty_cache()
    except: pass

RUN_TRAINING = True
BENCHMARK_VARIANTS = ["yolov11n", "yolov11s", "yolov11m"]
WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

if IS_KAGGLE:
    KAGGLE_INPUT = "/kaggle/input/datasets/mariabouguettaya/dataset-indoor"
    DATA_YAML_ORIG = f"{KAGGLE_INPUT}/data.yaml"
    DATA_YAML = "/kaggle/working/data_kaggle.yaml"
    TEST_IMG_DIR = Path(f"{KAGGLE_INPUT}/test/images")
    import yaml
    if os.path.exists(DATA_YAML_ORIG):
        with open(DATA_YAML_ORIG, 'r') as f: y_data = yaml.safe_load(f)
        y_data['path'] = KAGGLE_INPUT
        y_data['train'] = "train/images"
        y_data['val'] = "valid/images"
        y_data['test'] = "test/images"
        with open(DATA_YAML, 'w') as f: yaml.dump(y_data, f)
else:
    DATA_YAML = "../data_indoor_balanced.yaml"
    TEST_IMG_DIR = Path("../dataset_indoor_yolo_new/test/images")

IMG_SIZE = 640
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"
PATIENCE = 20
RESULTS_CSV  = "benchmark_yolov11_indoor.csv"
PERCLASS_CSV = "benchmark_yolov11_indoor_perclass.csv"
CLASS_NAMES = ["chair", "clock", "exit", "fireextinguisher", "printer", "screen", "trashbin"]

MODELS = [
    "yolov11n.pt",
    "yolov11s.pt",
    "yolov11m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


In [ ]:
import cv2
try:
    from sahi.predict import get_sliced_prediction
    from sahi import AutoDetectionModel
except:
    pass

def preprocess_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    final = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
    return final

def benchmark_model(model_name):
    print(f'\n{"="*60}\n  BENCHMARK: {model_name}\n{"="*60}')
    model = YOLO(model_name)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, workers=WORKERS, name=f"{model_name.replace('.pt','')}_ood", patience=PATIENCE, save=True)
    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=DATA_YAML, split='test', device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS)
    
    # Enhanced inference loop
    test_img_dir = TEST_IMG_DIR
    test_images = [p for p in test_img_dir.glob('*.*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}][:200]
    sahi_model = None
    if USE_SAHI:
        device_str = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'
        sahi_model = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(best_path), device=device_str, confidence_threshold=0.25)
    
    tn, fp, neg_total, lats = 0, 0, 0, []
    for img_p in test_images:
        inp = preprocess_image(img_p) if USE_PREPROCESSING else str(img_p)
        t0 = time.perf_counter()
        if USE_SAHI and sahi_model:
            res = get_sliced_prediction(inp, sahi_model, slice_height=SAHI_SLICE_SIZE, slice_width=SAHI_SLICE_SIZE, overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP, verbose=0)
            cnt = len(res.object_prediction_list)
        else:
            res = best_model(inp, verbose=False)
            cnt = len(res[0].boxes) if res[0].boxes else 0
        lats.append((time.perf_counter()-t0)*1000)
        lbl_p = Path(str(img_p).replace('images', 'labels')).with_suffix('.txt')
        if not lbl_p.exists() or lbl_p.stat().st_size == 0:
            neg_total += 1
            if cnt == 0: tn += 1
            else: fp += 1
    
    row = {
        'model': model_name.replace('.pt',''),
        'mAP@0.5': round(float(metrics.box.map50), 4),
        'speed_ms/img': round(float(np.mean(lats)), 2),
        'neg_acc': round(tn/neg_total if neg_total > 0 else 1.0, 4),
        'size_MB': round(best_path.stat().st_size/1e6, 1),
    }
    del model, best_model
    gc.collect(); _safe_cuda_empty_cache()
    return row, {}


In [ ]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — full training for each entry in MODELS.\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
            print("  If CUDA errors persist: restart the kernel, set WORKERS=0 or AMP=False in the config cell, then rerun.")
else:
    print("RUN_TRAINING=False — building table from existing best.pt (no training).\n")
    print(f"Resolved runs/detect -> {_runs_detect_dir()}\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no matching best.pt under {_runs_detect_dir()} (expect folder '{variant}' or '{variant}_*')")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")

## 1. Training Visualization

### 1.1 YOLOv11n Indoor (Nano)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path("../runs/detect/yolo11n_indoor")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]

print("Displaying YOLOv11n Indoor Training Results...")
for img_name in images:
    img_path = results_dir / img_name
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(img_path)))
        plt.axis("off")
        plt.title(f"YOLOv11n - {img_name}")
        plt.show()


### 1.2 YOLOv11s Indoor (Small)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path("../runs/detect/yolo11s_indoor")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]

print("Displaying YOLOv11s Indoor Training Results...")
for img_name in images:
    img_path = results_dir / img_name
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(img_path)))
        plt.axis("off")
        plt.title(f"YOLOv11s - {img_name}")
        plt.show()


### 1.3 YOLOv11m Indoor (Medium)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path("../runs/detect/yolo11m_indoor")
images = ["results.png", "confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"]

print("Displaying YOLOv11m Indoor Training Results...")
for img_name in images:
    img_path = results_dir / img_name
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(mpimg.imread(str(img_path)))
        plt.axis("off")
        plt.title(f"YOLOv11m - {img_name}")
        plt.show()


## 2. Model Performance Benchmarks

In [ ]:
RESULTS_CSV = "benchmark_yolov11_indoor.csv"
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PERCLASS_CSV = "benchmark_yolov11_indoor_perclass.csv"

_pcsv = Path(PERCLASS_CSV)
if not _pcsv.is_file() or _pcsv.stat().st_size < 5:
    print("No per-class CSV yet. Run the **build benchmark** cell first.")
else:
    df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)
    if df_pc.empty:
        print("Per-class table is empty.")
    else:
        print("Per-class mAP@0.5 across YOLOv11 variants")
        print("-" * 60)
        styled_pc = (
            df_pc.style
            .set_caption("Per-Class mAP@0.5 — YOLOv11 Benchmark")
            .format("{:.4f}")
            .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
            .set_properties(**{"text-align": "center", "font-size": "12px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
            ])
        )
        display(styled_pc)

In [ ]:
import pandas as pd
RESULTS_CSV = "benchmark_yolov11_indoor.csv"
import matplotlib.pyplot as plt
from pathlib import Path

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV — run the build-benchmark cell first.")
else:
    df = pd.read_csv(RESULTS_CSV)
    if df.empty or "model" not in df.columns:
        print("Results CSV has no data rows — skip chart.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
        axes[0].set_xlabel("mAP@0.5")
        axes[0].set_title("Accuracy (mAP@0.5)")
        axes[0].set_xlim(0, 1)
        axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
        axes[1].set_xlabel("ms / image")
        axes[1].set_title("Inference Speed")
        axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
        axes[2].set_xlabel("MB")
        axes[2].set_title("Model Size")
        plt.suptitle("YOLOv11 Benchmark — Indoor Dataset", fontsize=16, fontweight="bold")
        plt.tight_layout()
        plt.savefig("benchmark_yolov11_chart.png", dpi=150, bbox_inches="tight")
        plt.show()